# NHS England Maternal Care Pathways
## Stage 5.1 - Postpartum Emergency Readmissions

### Compiled by Federica Caretta Cortegiani

In [0]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last, split, flatten, contains, array_remove, filter
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import GeneralizedLinearRegression as GLR

%run "/Workspace/Shared/Helper_Methods_EP"


In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name = "msds_processed_cleaned_vars_stage"

df_master = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name}")

In [0]:
df_master_preg_id = df_master.select("UniqPregID", "person_id_deid", "ld_disch_date")

In [0]:
df_hes_ae = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.hes_ae_all_years "
) # PERSON_ID_DEID, AEKEY, ARRIVALDATE
df_hes_apc = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.hes_apc_all_years "
)  # EPIKEY, AEKEY, ADMIDATE, PERSON_ID_DEID
df_lowlat_ecds = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.lowlat_ecds_all_years "
) # PERSON_ID_DEID, ECDSKEY, ARRIVAL_DATE

In [0]:
# AE codes
df_hes_ae_agg = (
    df_hes_ae
    .groupBy("PERSON_ID_DEID", "ARRIVALDATE")
    .agg(
        collect_set("AEKEY").alias("AEKEYs"),
        flatten(array(*[collect_list(col(c)) for c in df_hes_ae.columns if (c.startswith("DIAG2") | c.startswith("DIAG3"))])).alias("ALL_DIAG_CODES")
    )
)

In [0]:
# ICD 10 (DIAG) and OPCS-4 (OPERTN)
df_hes_apc_agg = (
    df_hes_apc
    .withColumns({
        "DIAG_3_CONCAT": filter(split(col("DIAG_3_CONCAT"), ","), lambda x: ~x.startswith("-")),
        "DIAG_4_CONCAT": filter(split(col("DIAG_4_CONCAT"), ","), lambda x: ~x.startswith("-")),
        "OPERTN_3_CONCAT": filter(split(col("OPERTN_3_CONCAT"), ","), lambda x: ~x.startswith("-")),
        "OPERTN_4_CONCAT":  filter(split(col("OPERTN_4_CONCAT"), ","), lambda x: ~x.startswith("-"))
    })
    .groupBy("PERSON_ID_DEID", "ADMIDATE")
    .agg(
        collect_set("EPIKEY").alias("EPIKEYs"),
        flatten(
            array_union(array_union(collect_list("DIAG_3_CONCAT"), collect_list("DIAG_4_CONCAT")), array_union(collect_list("OPERTN_3_CONCAT"), collect_list("OPERTN_4_CONCAT")))
        ).alias("ALL_DIAG_CODES")
    )  
)

In [0]:
# SNOMED codes
df_lowlat_ecds_agg = (
    df_lowlat_ecds
    .groupBy("PERSON_ID_DEID", "ARRIVAL_DATE")
    .agg(
        collect_set("ECDSKEY").alias("ECDSKEYs"),
        array([collect_list(col(c)) for c in df_hes_ae.columns if c.startswith("DIAGNOSIS_CODE")]).alias("ALL_DIAG_CODES")
    )
)

In [0]:
df_hes_ae_agg_postpartum = (
    df_hes_ae_agg
    .join(
        df_master_preg_id,
        df_hes_ae_agg["PERSON_ID_DEID"] == df_master_preg_id["person_id_deid"],
        "inner"
    )
    .filter((datediff("ARRIVALDATE", "ld_disch_date") > 0) & (datediff("ARRIVALDATE", "ld_disch_date") <= 42))
    .drop("person_id_deid")
)

In [0]:
df_hes_apc_agg_postpartum = (
    df_hes_apc_agg
    .join(
        df_master_preg_id,
        df_hes_apc_agg["PERSON_ID_DEID"] == df_master_preg_id["person_id_deid"],
        "inner"
    )
    .filter((datediff("ADMIDATE", "ld_disch_date") > 0) & (datediff("ADMIDATE", "ld_disch_date") <= 42))
    .drop("person_id_deid")
)

In [0]:
df_lowlat_ecds_agg_postpartum = (
    df_lowlat_ecds_agg
    .join(
        df_master_preg_id,
        df_lowlat_ecds_agg["PERSON_ID_DEID"] == df_master_preg_id["person_id_deid"],
        "inner"
    )
    .filter((datediff("ARRIVAL_DATE", "ld_disch_date") > 0) & (datediff("ARRIVAL_DATE", "ld_disch_date") <= 42))
    .drop("person_id_deid")
)

In [0]:
hes_ae_codes_desc = spark.read.table("reference_data.dss_corporate.hes_ae_diag")
icd10_codes_desc = spark.read.table("reference_data.dss_corporate.icd10_codes")
snomed_codes_desc = spark.read.table("reference_data.dss_corporate.snomed_ct_rf2_descriptions")
opcs4_codes_desc = spark.read.table("reference_data.dss_corporate.opcs_codes_v02")

In [0]:
def readmission_codes(desc_col):
    single_word = [
        "puerperium", "postpartum", "postnatal", "puerperal", "perinatal", "Haemorrhage", "Hypertension", "Mastitis", "anaemia", "lactation", "obstetric", "anxiety", "depression", "preeclampsia", "eclampsia", "embolism", "preeclampsia", "pregnancy", "delivery", "labour", "oedema", "proteinuria"
    ]
    word_sets = [
        ["epidural", "anaesthesia"], ["spinal", "anaesthesia"], ["urinary", "infection"], ["breast", "inflammation"], ["venous", "thromboembolism"], ["perineal", "laceration"], ["medical", "misadventure"], ["digestive", "puerperium"], ["respiratory", "puerperium"], ["mental", "conditions"], ["skin", "subcutaneous", "puerperium"], ["gestational", "diabetes"], ["bipolar", "disorders"], ["liver", "puerperium"], ["bowel", "function"]
    ]

    single_cond = F.lit(False)
    if single_word:
        single_pattern = r"(?i)\b(" + "|".join(single_word) + r")\b"
        single_cond = col(desc_col).rlike(single_pattern)
        
    sets_cond = F.lit(False)
    for ws in word_sets:
        ws_cond = F.lit(True)
        for w in ws:
            ws_cond = ws_cond & col(desc_col).rlike(fr"(?i)\b{w}\b")
        sets_cond = sets_cond | ws_cond

    is_post = single_cond | sets_cond

    return is_post    

In [0]:
hes_ae_codes_desc_filt = hes_ae_codes_desc.filter(readmission_codes("AE_DIAG_DESCRIPTION"))

icd10_codes_desc_filt = icd10_codes_desc.filter(readmission_codes("DESCRIPTION"))

snomed_codes_desc_filt = snomed_codes_desc.filter(readmission_codes("TERM"))

opcs4_codes_desc_filt = opcs4_codes_desc.filter(readmission_codes("OPCS_CODE_DESC_FULL"))


In [0]:
hes_ae_codes_desc_list = hes_ae_codes_desc_filt.select("AE_DIAG_CODE", "AE_DIAG_DESCRIPTION").collect()

code_list = [row["AE_DIAG_CODE"] for row in hes_ae_codes_desc_list]
code_desc_list = []
for r in hes_ae_codes_desc_list:
    code_desc_list.extend([r["AE_DIAG_CODE"], r["AE_DIAG_DESCRIPTION"]])
desc_map = F.create_map(
    *[
        F.lit(x) for x in code_desc_list
    ]
)

df_hes_ae_agg_postpartum_readm = (
    df_hes_ae_agg_postpartum
    .withColumn("ALL_DIAG_CODES",  filter(col("ALL_DIAG_CODES"), lambda x: x.isin(code_list)))
    .withColumn("ALL_DIAG_DESCR", F.transform(col("ALL_DIAG_CODES"), lambda x: desc_map[x]))
    .filter(F.size(col("ALL_DIAG_CODES")) > 0)
)

In [0]:
icd10_codes_desc_list = icd10_codes_desc_filt.select("ALT_CODE", "DESCRIPTION").dropDuplicates(["ALT_CODE"]).collect()
opcs4_codes_desc_list = opcs4_codes_desc_filt.select("ALT_OPCS_CODE", "OPCS_CODE_DESC_FULL").dropDuplicates(["ALT_OPCS_CODE"]).collect()

code_list = [row["ALT_CODE"] for row in icd10_codes_desc_list] + [row["ALT_OPCS_CODE"] for row in opcs4_codes_desc_list]
code_desc_list = []
for r in icd10_codes_desc_list:
    code_desc_list.extend([r["ALT_CODE"], r["DESCRIPTION"]])
for r in opcs4_codes_desc_list:
    code_desc_list.extend([r["ALT_OPCS_CODE"], r["OPCS_CODE_DESC_FULL"]])
desc_map = F.create_map(
    *[
        F.lit(x) for x in code_desc_list
    ]
)

df_hes_apc_agg_postpartum_readm = (
    df_hes_apc_agg_postpartum
    .withColumn("ALL_DIAG_CODES",  filter(col("ALL_DIAG_CODES"), lambda x: x.isin(code_list)))
    .withColumn("ALL_DIAG_DESCR", F.transform(col("ALL_DIAG_CODES"), lambda x: desc_map[x]))
    .filter(F.size(col("ALL_DIAG_CODES")) > 0)
)

In [0]:
snomed_codes_desc_list = (
    snomed_codes_desc_filt
    .select("CONCEPT_ID", "TERM")
    .dropDuplicates(["CONCEPT_ID"])
    .collect()
)

code_list = [row["CONCEPT_ID"] for row in snomed_codes_desc_list]
code_desc_list = []
for r in snomed_codes_desc_list:
    code_desc_list.extend([r["CONCEPT_ID"], r["TERM"]])
desc_map = F.create_map(
    *[
        F.lit(x) for x in code_desc_list
    ]
)

df_lowlat_ecds_agg_postpartum_readm = (
    df_lowlat_ecds_agg_postpartum
    .withColumn("ALL_DIAG_CODES",  filter(col("ALL_DIAG_CODES"), lambda x: x.isin(code_list)))
    .withColumn("ALL_DIAG_DESCR", F.transform(col("ALL_DIAG_CODES"), lambda x: desc_map[x]))
    .filter(F.size(col("ALL_DIAG_CODES")) > 0)
)

In [0]:
uniq_preg_ids_hes_apc = [row["UniqPregID"] for row in df_hes_apc_agg_postpartum_readm.select("UniqPregID").distinct().collect()]
uniq_preg_ids_hes_ae = [row["UniqPregID"] for row in df_hes_ae_agg_postpartum_readm.select("UniqPregID").distinct().collect()]
uniq_preg_ids_lowlat_ecds = [row["UniqPregID"] for row in df_lowlat_ecds_agg_postpartum_readm.select("UniqPregID").distinct().collect()]

uniq_preg_ids = uniq_preg_ids_hes_apc + uniq_preg_ids_hes_ae + uniq_preg_ids_lowlat_ecds

df_master_with_readm = (
    df_master
    .withColumn("emergency_readmission", when(col("UniqPregID").isin(uniq_preg_ids), 1).otherwise(0))
)

In [0]:
%skip
print(f"Percentage: {100 * df_master_with_readm.filter(col("emergency_readmission") == 1).count() / df_master_with_readm.count()}%")

In [0]:
print(f"Number of rows: {'{:,}'.format(df_master_with_readm.count())}. Number of columns: {'{:,}'.format(len(df_master_with_readm.columns))}")

In [0]:
write_table_name = "msds_with_readmission_stage"

df_master_with_readm.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name}")